# Match Resolution Visualizer

Demonstrates the three-panel side-by-side visualizer for the llmr match-resolution pipeline:

1. **Before** — initial underspecified `Match` expression (free slots shown with dashed borders).
2. **World** — SymbolGraph grouped by class; bodies bound by the LLM are highlighted.
3. **After** — resolved `Match` expression; newly-filled slots are highlighted gold.

Binding arrows (dashed orange) connect each highlighted world body to its resolved leaf in the After panel.

> **Requirements:** `matplotlib`, `krrood`, `llmr`, a configured LLM (see `llm_backend_usage.ipynb`).

In [ ]:
# Standard setup — load .env for the API key if needed.
from dotenv import load_dotenv
load_dotenv(override=True)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

## 1. Build a minimal world

In [ ]:
from krrood.class_diagrams import ClassDiagram
from krrood.ontomatic.property_descriptor.attribute_introspector import DescriptorAwareIntrospector
from krrood.symbol_graph.symbol_graph import Symbol, SymbolGraph
from krrood.utils import recursive_subclasses
from dataclasses import dataclass
from typing_extensions import Optional

# Minimal scene object class
class WorldBody(Symbol):
    def __init__(self, name: str, parent: Optional['WorldBody'] = None) -> None:
        self.name = name

# Minimal action with one required entity slot
@dataclass
class PickUpAction:
    object_designator: WorldBody

# Initialise SymbolGraph and register scene objects
SymbolGraph.clear()
graph = SymbolGraph(
    _class_diagram=ClassDiagram(
        recursive_subclasses(Symbol),
        introspector=DescriptorAwareIntrospector(),
    )
)

milk  = WorldBody('milk_on_table')
table = WorldBody('table')
cup   = WorldBody('red_cup')
for obj in (milk, table, cup):
    graph.ensure_wrapped_instance(obj)

print('World ready:', [milk.name, table.name, cup.name])

## 2. Build a before-snapshot and a manually resolved after-snapshot

We'll skip the real LLM call here and resolve the slot by hand to keep the demo self-contained.

In [ ]:
from llmr.bridge.introspect import ActionFieldIntrospector
from llmr.bridge.match_reader import snapshot_match, underspecified_match, bind_slot_value
from llmr.schemas import (
    SlotFillingOutput, ActionAnnotationBundle, EntityDescription, SlotValue
)
from llmr.visualization.trees import MatchResolutionSnapshot

introspector = ActionFieldIntrospector()
match = underspecified_match(PickUpAction, introspector)

# Snapshot the 'before' state
snapshot = MatchResolutionSnapshot.from_match(match, symbol_type=WorldBody)

# Manually resolve object_designator → milk_on_table
data = snapshot_match(match, introspector)
bind_slot_value(data.slots[0], milk)

# Build a stub backend with semantics (mimics what LLMBackend._evaluate populates)
class StubBackend:
    semantics = ActionAnnotationBundle(
        action_type='PickUpAction',
        slot_filling=SlotFillingOutput(
            action_type='PickUpAction',
            slots=[
                SlotValue(
                    field_name='object_designator',
                    entity_description=EntityDescription(name='milk_on_table'),
                )
            ],
        ),
    )

snapshot.record_after(match, backend=StubBackend())
print('Newly filled:', snapshot.newly_filled)
print('Bindings:',     snapshot.bindings)

## 3. Render the 3-panel figure

In [ ]:
from llmr.visualization.render import render_panels

fig = render_panels(snapshot)
plt.show()

## 4. One-shot convenience function (with a real LLMBackend)

When using a live LLM the convenience function captures before/after automatically:

```python
from llmr.reasoning.llm_provider import make_llm, LLMProvider
from llmr.backend import LLMBackend
from llmr.bridge.match_reader import underspecified_match
from llmr.visualization.render import render_match_resolution

llm     = make_llm(LLMProvider.OPENAI)
backend = LLMBackend(llm=llm, symbol_type=WorldBody, instruction='pick up the milk')
match   = underspecified_match(PickUpAction)

# Evaluate and capture simultaneously
result = next(iter(backend.evaluate(match)))
fig    = render_match_resolution(match, backend, symbol_type=WorldBody)
fig.savefig('resolution.png', dpi=150)
```